In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass

# ========== 核心模型设定（严格遵守模型.pdf；参数留作可调） ==========
EARLY_CUTOFF = 12.0   # 早期阈值：t <= 12 周
LATE_CUTOFF  = 28.0   # 晚期阈值：t >= 28 周
T_MIN, T_MAX = 10.0, 30.0
BMI_MIN, BMI_MAX = 20.0, 45.0

def R_of_t(t: float) -> float:
    """分段风险函数：早期(<=12)低；中期(12,28)中；晚期(>=28)高。"""
    if t <= EARLY_CUTOFF:
        return 0.1
    elif t < LATE_CUTOFF:
        return 0.3
    else:
        return 1.0

def per_group_best_time(mu: float, sigma: float, lam: float, lower_bound: float):
    """
    给定组统计(均值mu, 标准差sigma)和惩罚权重lambda，求该组最佳t。
    目标: R(t) + lam * max(0, mu - t)^2, 约束: t ∈ [max(T_MIN, mu-2σ), T_MAX] 且 t ≥ lower_bound
    注：lower_bound 用于外部控制（一般取 max(T_MIN, mu - 2σ)）
    """
    lo = max(T_MIN, lower_bound)
    hi = T_MAX

    # 备选点：围绕关键断点与可行边界的有限枚举，保证稳健
    candidates = set()

    # 1) “零惩罚”最佳：t = clamp(mu, lo, hi)
    t_zero_pen = min(max(mu, lo), hi)
    candidates.add(float(t_zero_pen))

    # 2) 可行下界
    candidates.add(float(lo))

    # 3) 早期边界（希望进低风险区，但有“过早”二次惩罚）
    if lo <= EARLY_CUTOFF <= hi:
        candidates.add(float(EARLY_CUTOFF))

    # 4) 晚期边界（若mu或lo很大时可能不可避免）
    if lo <= LATE_CUTOFF <= hi:
        candidates.add(float(LATE_CUTOFF))

    # 5) 边界附近轻微扰动（避免等号处数值判断）——可选
    for eps in (1e-6,):
        if lo < EARLY_CUTOFF - eps < hi: candidates.add(EARLY_CUTOFF - eps)
        if lo < LATE_CUTOFF - eps < hi:  candidates.add(LATE_CUTOFF - eps)

    # 目标函数
    def obj(t):
        return R_of_t(t) + lam * max(0.0, mu - t)**2

    # 选最优
    best_t = None
    best_val = np.inf
    for t in candidates:
        val = obj(t)
        if val < best_val:
            best_val = val
            best_t = t
    return best_t, best_val

@dataclass
class GroupResult:
    idx_range: tuple
    N: int
    mu: float
    sigma: float
    t_star: float
    risk_term: float
    penalty_term: float
    obj: float
    bmi_lo: float
    bmi_hi: float

def compute_groups_and_times(df_sorted, cutpoints, lam, min_group_size):
    """
    给定已排序数据与分组 cutpoints（含 b0=20 与 b_n=45），计算各组统计与最优 t。
    返回：各组结果、总目标值
    """
    b = cutpoints
    results = []
    total_obj = 0.0
    n = len(b) - 1

    # 以 BMI 切分（左闭右开，最后一组右闭）
    bmi_vals = df_sorted["BMI"].values
    t_vals = df_sorted["earliest_time"].values

    # 找每个区间的索引范围
    starts = []
    ends = []
    start_idx = np.searchsorted(bmi_vals, b[0], side='left')
    for j in range(1, n):
        end_idx = np.searchsorted(bmi_vals, b[j], side='left')
        starts.append(start_idx)
        ends.append(end_idx)
        start_idx = end_idx
    # 最后一组
    starts.append(start_idx)
    ends.append(np.searchsorted(bmi_vals, b[-1], side='right'))

    for j in range(n):
        lo, hi = starts[j], ends[j]
        N_j = hi - lo
        if N_j < min_group_size:
            return None, np.inf  # 违反最小组容量约束，直接给无穷大

        times = t_vals[lo:hi]
        if N_j == 0:
            mu = np.nan
            sigma = np.nan
            t_star = np.nan
            val = np.inf
            risk_term = penalty_term = np.inf
        else:
            mu = float(np.mean(times))
            sigma = float(np.std(times, ddof=1)) if N_j > 1 else 0.0
            lower_bound = max(T_MIN, mu - 2.0 * sigma)
            t_star, _ = per_group_best_time(mu, sigma, lam, lower_bound)
            # 项分解
            risk_term = R_of_t(t_star)
            penalty_term = lam * max(0.0, mu - t_star)**2
            val = (risk_term + penalty_term) * N_j

        total_obj += val
        results.append(
            GroupResult(
                idx_range=(lo, hi),
                N=N_j, mu=mu, sigma=sigma, t_star=t_star,
                risk_term=risk_term, penalty_term=penalty_term, obj=val,
                bmi_lo=b[j], bmi_hi=b[j+1]
            )
        )
    return results, total_obj

def coordinate_descent_thresholds(df_in,
                                  n_groups=5,
                                  lam=0.1,
                                  min_group_frac=0.05,
                                  max_outer_iter=20,
                                  scan_stride=5,
                                  verbose=True):
    """
    离散坐标下降：在“数据 BMI 唯一值的中点”上扫描每一道切分点，交替更新分组与组内最优时点。
    - n_groups: 分组数 n
    - lam: 惩罚系数 λ
    - min_group_frac: 每组至少占总体比例（用于避免空组/极小组）
    - max_outer_iter: 最大外循环迭代次数
    - scan_stride: 扫描步长（降低复杂度）；数据量不大时可设为1
    """
    # 1) 数据过滤与排序（严格在 [20, 45] 的 BMI 可行域内）
    df = df_in.copy()
    df = df[np.isfinite(df["BMI"]) & np.isfinite(df["earliest_time"])]
    df = df[(df["BMI"] >= BMI_MIN) & (df["BMI"] <= BMI_MAX)].copy()
    df = df.sort_values("BMI").reset_index(drop=True)

    if len(df) < n_groups:
        raise ValueError("样本量小于分组数，无法分组。")

    N_total = len(df)
    min_group_size = max(1, int(np.floor(min_group_frac * N_total)))

    # 2) 初始 cutpoints：按 BMI 分位数初始化（包含两端点）
    quantiles = np.linspace(0, 1, n_groups + 1)
    b = [float(df["BMI"].quantile(q)) for q in quantiles]
    # 强制端点精确为模型规定
    b[0]  = BMI_MIN
    b[-1] = BMI_MAX

    # 保证严格递增（避免重复 quantile 导致的退化）
    for k in range(1, len(b)):
        if b[k] <= b[k-1]:
            b[k] = np.nextafter(b[k-1], float('inf'))

    # 3) 预计算候选切分位置（用 BMI 相邻唯一值的中点）
    uniq_bmi = np.unique(df["BMI"].values)
    mids = (uniq_bmi[:-1] + uniq_bmi[1:]) / 2.0
    if len(mids) == 0:
        raise ValueError("BMI 唯一值不足以形成切分。")

    # 外循环：坐标下降
    best_results, best_obj = compute_groups_and_times(df, b, lam, min_group_size)
    if best_obj == np.inf:
        # 量纲很极端时，用等频分位数微调一次
        pass

    for it in range(max_outer_iter):
        improved = False
        if verbose:
            print(f"[Iter {it+1}] Current objective = {best_obj:.6f}")

        # 依次优化每一道 cut（不含两端点）
        for cut_idx in range(1, n_groups):
            # cut 可选位置需在 (b[cut_idx-1], b[cut_idx+1]) 之间
            lo_allow = b[cut_idx-1]
            hi_allow = b[cut_idx+1]

            # 生成候选（控制步长以降复杂度）
            cand_pos = mids[(mids > lo_allow) & (mids < hi_allow)]
            cand_pos = cand_pos[::max(1, scan_stride)]
            # 始终包含当前 cut 位置
            cand_pos = np.concatenate([[b[cut_idx]], cand_pos])

            # 在满足“最小组容量”的前提下，逐一尝试
            local_best = best_obj
            local_best_cut = b[cut_idx]
            for cp in cand_pos:
                b_try = b.copy()
                b_try[cut_idx] = float(cp)

                # 组容量硬约束：快速排除极小组
                # 用索引区间估算组大小是否可能满足
                # 直接算一下结果（里面有更严格的检查）
                res_try, obj_try = compute_groups_and_times(df, b_try, lam, min_group_size)
                if obj_try < local_best:
                    local_best = obj_try
                    local_best_cut = float(cp)
                    local_best_res = res_try

            # 若该 cut 有改进，则接受
            if local_best + 1e-12 < best_obj:
                b[cut_idx] = local_best_cut
                best_obj = local_best
                best_results = local_best_res
                improved = True

        if not improved:
            if verbose:
                print(f"Converged at iter {it+1}. Objective = {best_obj:.6f}")
            break

    # 汇总结果为 DataFrame
    out_rows = []
    for j, gr in enumerate(best_results, start=1):
        out_rows.append({
            "group": j,
            "BMI_interval": f"[{gr.bmi_lo:.4f}, {gr.bmi_hi:.4f}" + ("]" if j==len(best_results) else ")"),
            "N": gr.N,
            "mu_earliest_time": gr.mu,
            "sigma_earliest_time": gr.sigma,
            "t_star": gr.t_star,
            "risk_R(t*)": gr.risk_term,
            "penalty_lambda*(mu - t*)^2": gr.penalty_term,
            "group_obj": gr.obj
        })
    summary_df = pd.DataFrame(out_rows)
    return {
        "cutpoints": b,
        "objective": best_obj,
        "group_table": summary_df.sort_values("group").reset_index(drop=True),
    }


# ===================== 使用示例 =====================
# 假设你已有 df，包含两列：'BMI' 和 'earliest_time'
# df = your_dataframe

# 示例运行（请根据需要调整超参数）
# result = coordinate_descent_thresholds(
#     df, n_groups=5, lam=0.1,
#     min_group_frac=0.05,  # 每组至少占 5%
#     max_outer_iter=20, scan_stride=3, verbose=True
# )
# print("最优 cutpoints（含端点）:", result["cutpoints"])
# print("最优目标值:", result["objective"])
# print(result["group_table"])


In [ ]:
df=pd.read_csv('BMI-earliest.csv')
result = coordinate_descent_thresholds(
    df, n_groups=5, lam=0.1,
    min_group_frac=0.05,  # 每组至少占 5%
    max_outer_iter=20, scan_stride=3, verbose=True
)
print("最优 cutpoints（含端点）:", result["cutpoints"])
print("最优目标值:", result["objective"])
print(result["group_table"])